# Lab 1: Hello, Robot! — The First Step to Autonomy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatayer8891-boop/zuj-robotics/blob/main/labs/lab-01-hello-robot/notebook.ipynb)

---

**Course:** Introduction to Robotics for AI and Data Science (0135343)  
**Instructor:** Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/)  
**University:** Al-Zaytoonah University of Jordan

---

## Overview

Spawn a robot in PyBullet, read its sensors, drive it in a square, and visualize the trajectory.

> **80/20 Principle:** Focus on understanding the core algorithm in each activity. The implementation details will follow naturally once you grasp the concept.


In [ ]:
# ============================================================
# COLAB ENVIRONMENT SETUP
# ============================================================
# This cell installs the required packages for running this lab
# in Google Colab. If you're running locally, you can skip this.
# ============================================================

import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Setting up Colab environment...")
    !pip install pybullet numpy matplotlib opencv-python-headless -q
    print("✅ All packages installed!")
    print("⚠️  Note: PyBullet runs in HEADLESS mode on Colab (no 3D GUI).")
    print("   You'll see matplotlib plots instead of the live 3D window.")
    print("   For the full 3D experience, run locally with: conda activate robotics_YourName")
else:
    print("✅ Running locally — full GUI mode available.")


---

## Activity 1: Spawn Robot

**File:** `spawn_robot.py`

spawn_robot_v2.py Description: This script serves as the most basic "Hello, World!" for robotics simulation. Its purpose is to demonstrate the fundamental steps of: 1. Connecting to the physics engine (PyBullet). 2. Loading the simulation environment (a ground plane). 3. Loading a robot model from a URDF file. 4. Running the simulation for a fixed duration. Every robotics simulation you build will start with these core steps.


In [ ]:
# spawn_robot_v2.py
#
# Description:
# This script serves as the most basic "Hello, World!" for robotics simulation.
# Its purpose is to demonstrate the fundamental steps of:
#   1. Connecting to the physics engine (PyBullet).
#   2. Loading the simulation environment (a ground plane).
#   3. Loading a robot model from a URDF file.
#   4. Running the simulation for a fixed duration.
#
# Every robotics simulation you build will start with these core steps.

import pybullet as p
import time
import pybullet_data

def main():
    """Initializes the simulation and spawns a robot."""
    
    # --- 1. Connect to the Physics Engine ---
    # p.GUI starts the simulation with a graphical user interface.
    # p.DIRECT would run the simulation without a window (useful for batch processing).
    print("Starting simulation...")
    physicsClient = p.connect(p.DIRECT if IN_COLAB else p.GUI)
    print(f"Connected to physics server with ID: {physicsClient}")

    # --- 2. Set up the Simulation Environment ---
    # Set the search path for PyBullet to find its default assets (like the plane).
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    
    # Set the gravity of the world. The default is (0, 0, -9.8).
    p.setGravity(0, 0, -9.8)
    
    # Load a ground plane. The URDF (Unified Robot Description Format) is an XML file
    # that describes all elements of a robot or, in this case, a simple plane.
    print("Loading ground plane...")
    planeId = p.loadURDF("plane.urdf")
    print(f"Loaded plane with ID: {planeId}")

    # --- 3. Load the Robot ---
    # Define the robot's starting position and orientation.
    # Position is in meters (x, y, z).
    # Orientation is a quaternion (x, y, z, w), which represents rotation.
    # p.getQuaternionFromEuler converts from human-readable Euler angles (roll, pitch, yaw).
    startPos = [0, 0, 0.1]  # Start slightly above the ground to avoid initial collisions.
    startOrientation = p.getQuaternionFromEuler([0, 0, 0])
    
    # Load the R2D2 model. PyBullet will search for this file in its data path.
    # The function returns a unique integer ID for the robot body.
    print("Loading robot model...")
    robotId = p.loadURDF("r2d2.urdf", startPos, startOrientation)
    print(f"Loaded robot with ID: {robotId}")

    # --- 4. Run the Simulation ---
    # The simulation runs in a loop. In each step, we advance the physics.
    # 1/240 is the default time step (240 Hz). The simulation will run for 5 seconds.
    print("Running simulation for 5 seconds...")
    for i in range(int(5 * 240)):
        p.stepSimulation()
        # time.sleep(1./240.) # Optional: slow down to real-time. Not needed for simple sims.

    print("Simulation finished.")

    # --- 5. Disconnect from the Physics Engine ---
    # It's good practice to clean up and close the connection.
    p.disconnect()
    print("Disconnected from physics server.")

if __name__ == "__main__":
    main()


---

## Activity 2: Read Sensors

**File:** `read_sensors.py`

read_sensors_v2.py Description: This script builds upon spawn_robot.py by introducing the "Sense" part of the Sense-Plan-Act cycle. Its purpose is to demonstrate how to query the physics engine for the state of a robot. Key Concepts: - Robot State: A robot's state includes its position, orientation, and velocities. - Proprioceptive Sensing: Sensing the internal state of the robot (as opposed to exteroceptive sensing, which is sensing the external world, like with a camera). - getBasePositionAndOrientation: The core PyBullet function for this task.


In [ ]:
# read_sensors_v2.py
#
# Description:
# This script builds upon spawn_robot.py by introducing the "Sense" part of the
# Sense-Plan-Act cycle. Its purpose is to demonstrate how to query the physics
# engine for the state of a robot.
#
# Key Concepts:
#   - Robot State: A robot's state includes its position, orientation, and velocities.
#   - Proprioceptive Sensing: Sensing the internal state of the robot (as opposed to
#     exteroceptive sensing, which is sensing the external world, like with a camera).
#   - getBasePositionAndOrientation: The core PyBullet function for this task.

import pybullet as p
import time
import pybullet_data

def main():
    """Initializes simulation, spawns a robot, and reads its sensor data."""
    
    # --- 1. Initialization (Same as spawn_robot.py) ---
    physicsClient = p.connect(p.DIRECT if IN_COLAB else p.GUI)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.8)
    planeId = p.loadURDF("plane.urdf")
    startPos = [0, 0, 0.1]
    startOrientation = p.getQuaternionFromEuler([0, 0, 0])
    robotId = p.loadURDF("r2d2.urdf", startPos, startOrientation)
    print(f"Robot with ID {robotId} spawned.")

    # --- 2. Simulation Loop with Sensing ---
    # We will run the simulation for 5 seconds and print the robot's state every 0.5 seconds.
    print("\n--- Reading Robot State for 5 seconds ---")
    for i in range(int(5 * 240)):
        p.stepSimulation()

        # We only print every 120 steps (0.5 seconds) to avoid flooding the console.
        if i % 120 == 0:
            # --- Get the robot's position and orientation ---
            # This is the core function for proprioceptive sensing of the robot's base.
            # It returns the position (x, y, z) and orientation (a quaternion).
            pos, orn = p.getBasePositionAndOrientation(robotId)
            
            # --- Get the robot's velocity ---
            # This function returns the linear velocity (vx, vy, vz) and angular velocity (wx, wy, wz).
            linear_vel, angular_vel = p.getBaseVelocity(robotId)

            # --- Print the results in a readable format ---
            print(f"Time: {i/240.0:.2f}s")
            print(f"  Position (x,y,z):    ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})")
            
            # Convert quaternion to Euler for easier interpretation
            orn_euler = p.getEulerFromQuaternion(orn)
            print(f"  Orientation (r,p,y): ({orn_euler[0]:.3f}, {orn_euler[1]:.3f}, {orn_euler[2]:.3f})")
            
            print(f"  Linear Velocity:     ({linear_vel[0]:.3f}, {linear_vel[1]:.3f}, {linear_vel[2]:.3f})")
            print(f"  Angular Velocity:    ({angular_vel[0]:.3f}, {angular_vel[1]:.3f}, {angular_vel[2]:.3f})\n")

    print("Simulation finished.")
    p.disconnect()

if __name__ == "__main__":
    main()


---

## Activity 3: Move Robot

**File:** `move_robot.py`

move_robot_v4.py Description: This script demonstrates a simpler, more direct way to control robot motion using resetBaseVelocity. Instead of controlling individual wheel joints, we command the robot's body (its "base") directly with linear and angular velocities. This is less physically realistic but much easier for beginners to understand and control. Key Concepts: - Base Control: Commanding the entire robot body's velocity. - resetBaseVelocity: A PyBullet function to directly set the linear and angular velocity of a robot's base link. - Body-to-World Frame Conversion: Converting desired forward/turn speeds into global X, Y, Z velocities based on the robot's current orientation.


In [ ]:
# move_robot_v4.py
#
# Description:
# This script demonstrates a simpler, more direct way to control robot motion
# using resetBaseVelocity. Instead of controlling individual wheel joints, we
# command the robot's body (its "base") directly with linear and angular velocities.
# This is less physically realistic but much easier for beginners to understand and control.
#
# Key Concepts:
#   - Base Control: Commanding the entire robot body's velocity.
#   - resetBaseVelocity: A PyBullet function to directly set the linear and angular
#     velocity of a robot's base link.
#   - Body-to-World Frame Conversion: Converting desired forward/turn speeds into
#     global X, Y, Z velocities based on the robot's current orientation.

import pybullet as p
import time
import pybullet_data
import math

def move_square(robotId):
    """Drives the robot in a 2x2 meter square using resetBaseVelocity."""
    
    # --- Define Movement Parameters ---
    side_length = 2.0  # meters
    linear_speed = 1.0 # m/s
    turn_speed_rad = math.pi / 2 # 90 deg/s in radians

    # --- Calculate Durations ---
    move_duration = side_length / linear_speed # 2 seconds
    turn_duration = (math.pi / 2) / turn_speed_rad # 1 second for a 90-degree turn

    print("\n--- Starting to move in a square ---")
    for i in range(4):
        print(f"Side {i+1}/4: Moving forward...")
        # --- Move Forward ---
        for _ in range(int(move_duration * 240)):
            _, orientation = p.getBasePositionAndOrientation(robotId)
            euler = p.getEulerFromQuaternion(orientation)
            yaw = euler[2]
            world_vx = linear_speed * math.cos(yaw)
            world_vy = linear_speed * math.sin(yaw)
            p.resetBaseVelocity(robotId, linearVelocity=[world_vx, world_vy, 0], angularVelocity=[0, 0, 0])
            p.stepSimulation()
            time.sleep(1./240.)

        print(f"Side {i+1}/4: Turning left...")
        # --- Turn Left ---
        for _ in range(int(turn_duration * 240)):
            p.resetBaseVelocity(robotId, linearVelocity=[0, 0, 0], angularVelocity=[0, 0, turn_speed_rad])
            p.stepSimulation()
            time.sleep(1./240.)

    # --- Stop the robot ---
    print("Finished square. Stopping robot.")
    p.resetBaseVelocity(robotId, linearVelocity=[0, 0, 0], angularVelocity=[0, 0, 0])
    for _ in range(240):
        p.stepSimulation()

def main():
    """Initializes simulation, spawns a robot, and commands it to move."""
    
    physicsClient = p.connect(p.DIRECT if IN_COLAB else p.GUI)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.8)
    planeId = p.loadURDF("plane.urdf")
    
    # Using R2D2 because it works well with resetBaseVelocity
    startPos = [0, 0, 0.5]
    startOrientation = p.getQuaternionFromEuler([0, 0, 0])
    robotId = p.loadURDF("r2d2.urdf", startPos, startOrientation)
    print(f"R2D2 robot with ID {robotId} spawned.")

    move_square(robotId)

    print("Simulation finished.")
    p.disconnect()

if __name__ == "__main__":
    main()


---

## Activity 4: Visualize Trajectory

**File:** `visualize_trajectory.py`

visualize_trajectory_v4.py Description: This script combines all previous concepts and adds data analysis and visualization. It runs the simulation in the background (DIRECT mode) for speed, records the robot's trajectory, and then uses Matplotlib to plot the results. This is a critical skill for debugging and verifying robot performance. Key Concepts: - DIRECT Mode: Running simulation without GUI for faster-than-real-time execution. - Data Logging: Storing state information at each time step for later analysis. - Visualization: Using plots to understand and validate robot behavior.


In [ ]:
# visualize_trajectory_v4.py
#
# Description:
# This script combines all previous concepts and adds data analysis and visualization.
# It runs the simulation in the background (DIRECT mode) for speed, records the
# robot's trajectory, and then uses Matplotlib to plot the results.
# This is a critical skill for debugging and verifying robot performance.
#
# Key Concepts:
#   - DIRECT Mode: Running simulation without GUI for faster-than-real-time execution.
#   - Data Logging: Storing state information at each time step for later analysis.
#   - Visualization: Using plots to understand and validate robot behavior.

import pybullet as p
import pybullet_data
import numpy as np
import matplotlib.pyplot as plt
import math
import sys
IN_COLAB = 'google.colab' in sys.modules



def run_simulation_and_get_trajectory():
    """Runs the simulation in DIRECT mode and returns the trajectory data."""
    
    # --- 1. Initialization in DIRECT mode ---
    # DIRECT mode runs without a GUI window, which is much faster.
    # This is ideal for batch processing, data collection, and automated testing.
    physicsClient = p.connect(p.DIRECT)
    p.setAdditionalSearchPath(pybullet_data.getDataPath())
    p.setGravity(0, 0, -9.8)
    p.loadURDF("plane.urdf")
    robotId = p.loadURDF("r2d2.urdf", [0, 0, 0.5], p.getQuaternionFromEuler([0, 0, 0]))
    print("Simulation started in DIRECT mode...")

    # --- 2. Data Logging Setup ---
    # We'll store the robot's position at each step in a list.
    trajectory = []

    # --- 3. Execute Movement (same logic as move_robot.py) ---
    side_length = 2.0
    linear_speed = 1.0
    turn_speed_rad = math.pi / 2

    move_duration = side_length / linear_speed
    turn_duration = (math.pi / 2) / turn_speed_rad

    for i in range(4):
        # Move Forward
        for _ in range(int(move_duration * 240)):
            _, orientation = p.getBasePositionAndOrientation(robotId)
            euler = p.getEulerFromQuaternion(orientation)
            yaw = euler[2]
            world_vx = linear_speed * math.cos(yaw)
            world_vy = linear_speed * math.sin(yaw)
            p.resetBaseVelocity(robotId, linearVelocity=[world_vx, world_vy, 0], angularVelocity=[0, 0, 0])
            pos, _ = p.getBasePositionAndOrientation(robotId)
            trajectory.append(pos)
            p.stepSimulation()

        # Turn Left
        for _ in range(int(turn_duration * 240)):
            p.resetBaseVelocity(robotId, linearVelocity=[0, 0, 0], angularVelocity=[0, 0, turn_speed_rad])
            pos, _ = p.getBasePositionAndOrientation(robotId)
            trajectory.append(pos)
            p.stepSimulation()

    p.disconnect()
    print("Simulation finished. Trajectory recorded.")
    return np.array(trajectory)

def analyze_and_plot_trajectory(trajectory):
    """Takes trajectory data and creates plots for analysis."""
    print("Analyzing and plotting trajectory...")
    
    # --- 1. Create a time vector ---
    time = np.arange(len(trajectory)) / 240.0

    # --- 2. Create the Main Plot Figure ---
    # We create a figure with 2x2 subplots for detailed analysis.
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Robot Trajectory Analysis", fontsize=16)

    # --- Plot 1: Top-Down Trajectory (X vs Y) ---
    # This is the most important plot. A correct square should be clearly visible.
    ax = axs[0, 0]
    ax.plot(trajectory[:, 0], trajectory[:, 1], label="Robot Path")
    ax.plot(trajectory[0, 0], trajectory[0, 1], "go", markersize=10, label="Start")
    ax.plot(trajectory[-1, 0], trajectory[-1, 1], "rs", markersize=10, label="End")
    ax.set_title("Top-Down Trajectory")
    ax.set_xlabel("X Position (m)")
    ax.set_ylabel("Y Position (m)")
    ax.grid(True)
    ax.axis("equal")
    ax.legend()

    # --- Plot 2: X Position vs Time ---
    ax = axs[0, 1]
    ax.plot(time, trajectory[:, 0], label="X Position")
    ax.set_title("X Position over Time")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("X Position (m)")
    ax.grid(True)
    ax.legend()

    # --- Plot 3: Y Position vs Time ---
    ax = axs[1, 0]
    ax.plot(time, trajectory[:, 1], label="Y Position")
    ax.set_title("Y Position over Time")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Y Position (m)")
    ax.grid(True)
    ax.legend()

    # --- Plot 4: Z Position vs Time (for stability check) ---
    # If the robot is stable, Z should remain roughly constant.
    ax = axs[1, 1]
    ax.plot(time, trajectory[:, 2], label="Z Position")
    ax.set_title("Z Position (Height) over Time")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Z Position (m)")
    ax.grid(True)
    ax.legend()
    ax.set_ylim(0, 1.0)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig("robot_trajectory_analysis.png")
    print("Saved detailed analysis plot to robot_trajectory_analysis.png")
    plt.show()

def main():
    """Main function to run simulation and visualize results."""
    trajectory_data = run_simulation_and_get_trajectory()
    analyze_and_plot_trajectory(trajectory_data)

if __name__ == "__main__":
    main()


---

## Summary & Next Steps

Congratulations on completing this lab! Before moving on:

1. **Review** your outputs and make sure they match expected behavior
2. **Experiment** by modifying parameters and observing the effects
3. **Reflect** on what you learned — write a brief paragraph in your report

---

*Dr. Akram Fatayer · [a.fatayer@zuj.edu.jo](mailto:a.fatayer@zuj.edu.jo) · [LinkedIn](https://www.linkedin.com/in/akram-fatayer/) · Al-Zaytoonah University of Jordan*
